# Gate 5G: behavioral fidelity and ranking splicing

Thin interface to the standalone manual pipeline. No training or attribution is performed. See `GATE5G_RUNBOOK.md` for execution, code locks, statistical definitions and recovery. Registered GPU evaluations are disabled by default. No Gate 5G prompt may receive a model forward before freezing.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/mattr_fidelity_splice').is_dir())
OUT = ROOT / 'results/mattr_fidelity_splice/gate5g'

def stage(name, *args, execute=False):
    cmd = [sys.executable, '-u', '-m', 'src.mattr_fidelity_splice.cli', name, '--out', str(OUT), *args]
    print(' '.join(cmd))
    if execute:
        subprocess.run(cmd, cwd=ROOT, check=True)

print(ROOT)
print('Use the existing pinned WSL kernel. All execution flags default to False.')

## Historical-only preflight

Reconstructs original signed EAP-IG ordering and verifies topology/mask hashes before any fresh inference. The bounded GPU test uses two excluded historical prompts. Preflight never freezes implicitly.

In [ ]:
RUN_PREFLIGHT = False
stage('preflight', execute=RUN_PREFLIGHT)

In [ ]:
for name in ['ranking_inventory.csv', 'splice_topology.csv', 'cohort_hashes.csv']:
    if (OUT / name).exists():
        print(name)
        display(pd.read_csv(OUT / name).head())

## Freeze after the implementation commit

Requires clean tracked sources and successful correctness checks. The immutable protocol references the implementation commit; do not make another commit until registered evaluation has completed.

In [ ]:
FREEZE_PROTOCOL = False
stage('freeze', execute=FREEZE_PROTOCOL)

## Registered inference — manual opt-in only

All CLI jobs independently enforce the frozen code/protocol/ranking/cohort lock. Run A before B/C to calibrate the five-hour runtime projection. Repeat identical commands to resume verified mask/job outputs. Add `--approve-over-5h` only after reviewing the measured projection.

In [ ]:
RUN_PART_A = False
stage('fidelity', '--job', 'baseline_n10000_s42_step5000', execute=RUN_PART_A)

In [ ]:
RUN_PART_B = False
stage('diagnose', execute=RUN_PART_B)

In [ ]:
RUN_PART_C = False
stage('splice', '--method', 'baseline', '--seed', '42', '--k', '324', execute=RUN_PART_C)

In [ ]:
RUN_ALL = False
stage('run-all', execute=RUN_ALL)

## Offline aggregation and audit

Requires complete registered jobs. Uses shared exact-template paired prompt bootstraps and the frozen A/C decisions. This stage never runs model forwards. The researcher interpretation remains empty.

In [ ]:
RUN_ANALYSIS = False
RUN_AUDIT = False
stage('analyze', execute=RUN_ANALYSIS)
stage('audit', execute=RUN_AUDIT)

In [ ]:
for name in ['part_a_registered_comparisons.csv', 'cap116_fidelity_per_seed.csv', 'splice_paired_differences.csv', 'compute_accounting.csv']:
    if (OUT / name).exists():
        print(name)
        display(pd.read_csv(OUT / name))
if (OUT / 'gate5g_decision.json').exists():
    print(json.dumps(json.loads((OUT / 'gate5g_decision.json').read_text()), indent=2))

## Review and package

Fill the researcher-written interpretation in the generated campaign report after inspecting actual results. Use `07_package.sh --dry-run`, then `07_package.sh`, review staging, then `07_package.sh --commit`. Push manually. Raw prompts, full-vocabulary reference caches, model weights and tensor scores stay local. Stop after Gate 5G.